# Fase 1 · M04c: Corrección de Notas de Acceso

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 1 — Transformación |
| **Módulo** | M04c — Corrección Notas |

---

## 🎯 Qué hace

Corrige los valores incorrectos en la variable nota_acceso cruzando con la tabla de preinscripción.

## 📋 Requisitos

- `data/02_processed/df_alumno.parquet`
- `data/01_interim/preinscripcion.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/02_processed/df_alumno.parquet` | Dataset con notas de acceso corregidas (sobreescribe) |

## 🔄 Flujo

```
df_alumno.parquet + preinscripcion.parquet
    ↓ Detección de notas incorrectas
    ↓ Cruce con preinscripción
    → df_alumno.parquet (corregido)
```

## ➡️ Siguiente

`f1_m04d_correccion_via_acceso.ipynb` — corrección de vía de acceso


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# --- Detectar entorno ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import RUTA_INTERIM, RUTA_PROCESSED, info_entorno
from src.constantes import UJI_ID

# --- Info entorno ---
info_entorno()
print(f'\n📂 Rutas:')
print(f'   Interim:   {RUTA_INTERIM}')
print(f'   Processed: {RUTA_PROCESSED}')

✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ============================

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS')
print('=' * 60)

# --- Dataset principal ---
path_alumno = RUTA_PROCESSED / 'df_alumno.parquet'
if not path_alumno.exists():
    raise FileNotFoundError(
        f'No encontrado: {path_alumno}\n'
        f'Ejecuta primero f1_m04b_union_preinscripcion.ipynb'
    )

df = pd.read_parquet(path_alumno)
print(f'✅ df_alumno: {len(df):,} registros × {len(df.columns)} columnas')

# --- Preinscripción ---
path_pre = RUTA_INTERIM / 'preinscripcion.parquet'
if not path_pre.exists():
    raise FileNotFoundError(f'No encontrado: {path_pre}')

df_pre = pd.read_parquet(path_pre)
print(f'✅ preinscripcion: {len(df_pre):,} registros')

CARGANDO DATOS


✅ df_alumno: 109,568 registros × 37 columnas


✅ preinscripcion: 210,986 registros


In [3]:
# ============================================================================
# CELDA 3: DIAGNÓSTICO — ESTADO ACTUAL DE LAS NOTAS
# ============================================================================

print('=' * 60)
print('DIAGNÓSTICO: ESTADO ACTUAL')
print('=' * 60)

n_total = len(df)

# --- nota_acceso ---
na_acceso = df['nota_acceso'].isna().sum()
print(f'\n📊 nota_acceso:')
print(f'   Con valor:  {n_total - na_acceso:,} ({(n_total - na_acceso)/n_total*100:.1f}%)')
print(f'   NaN:        {na_acceso:,} ({na_acceso/n_total*100:.1f}%)')
print(f'   Rango:      [{df["nota_acceso"].min():.4f}, {df["nota_acceso"].max():.4f}]')
gt14_acceso = (df['nota_acceso'] > 14).sum()
print(f'   Valores >14: {gt14_acceso}')

# --- nota_selectividad ---
na_sel = df['nota_selectividad'].isna().sum()
gt14_sel = (df['nota_selectividad'] > 14).sum()
print(f'\n📊 nota_selectividad:')
print(f'   Con valor:  {n_total - na_sel:,} ({(n_total - na_sel)/n_total*100:.1f}%)')
print(f'   NaN:        {na_sel:,} ({na_sel/n_total*100:.1f}%)')
print(f'   Rango:      [{df["nota_selectividad"].min():.4f}, {df["nota_selectividad"].max():.4f}]')
print(f'   ⚠️ Valores >14: {gt14_sel}')
if gt14_sel > 0:
    print(f'   Detalle >14: {df.loc[df["nota_selectividad"] > 14, "nota_selectividad"].value_counts().to_dict()}')

# --- Sin ninguna nota ---
sin_ninguna_antes = (df['nota_acceso'].isna() & df['nota_selectividad'].isna()).sum()
print(f'\n📊 Sin NINGUNA nota (ni acceso ni selectividad):')
print(f'   {sin_ninguna_antes:,} registros ({sin_ninguna_antes/n_total*100:.1f}%)')

# Guardar métricas para comparación final
antes = {
    'nota_acceso_nan': na_acceso,
    'nota_selectividad_nan': na_sel,
    'nota_selectividad_gt14': gt14_sel,
    'sin_ninguna': sin_ninguna_antes
}

DIAGNÓSTICO: ESTADO ACTUAL

📊 nota_acceso:
   Con valor:  98,595 (90.0%)
   NaN:        10,973 (10.0%)
   Rango:      [5.0000, 13.8050]
   Valores >14: 0

📊 nota_selectividad:
   Con valor:  95,314 (87.0%)
   NaN:        14,254 (13.0%)
   Rango:      [5.0000, 7364.0000]
   ⚠️ Valores >14: 5
   Detalle >14: {7364.0: 5}

📊 Sin NINGUNA nota (ni acceso ni selectividad):
   10,747 registros (9.8%)


In [4]:
# ============================================================================
# CELDA 4: DIAGNÓSTICO — NOTAS EN PREINSCRIPCIÓN
# ============================================================================
# La tabla de preinscripción tiene 2 columnas de nota con errores de escala:
#   - nota_txt (float): nota de acceso → para rellenar nota_acceso
#   - nota_1   (int):   nota fase general → para rellenar nota_selectividad
# Muchos valores están multiplicados ×10, ×100 o ×1000 (error del sistema).
# ============================================================================

print('=' * 60)
print('DIAGNÓSTICO: NOTAS EN PREINSCRIPCIÓN')
print('=' * 60)

# Filtrar solo UJI
pre_uji = df_pre[df_pre['uni_id'] == UJI_ID].copy()
print(f'\nRegistros UJI: {len(pre_uji):,} de {len(df_pre):,} totales')

for col, desc in [('nota_txt', 'nota de acceso'), ('nota_1', 'nota selectividad')]:
    vals = pd.to_numeric(pre_uji[col], errors='coerce').astype(float)
    gt14 = (vals > 14).sum()
    print(f'\n📊 {col} ({desc}):')
    print(f'   Rango: [{vals.min():.1f}, {vals.max():.1f}]')
    print(f'   ⚠️ Valores >14: {gt14:,}')
    print(f'   Distribución por escala:')
    print(f'     5-14   (correctos):  {((vals >= 5) & (vals <= 14)).sum():>7,}')
    print(f'     15-140  (÷10):       {((vals > 14) & (vals <= 140)).sum():>7,}')
    print(f'     141-1400 (÷100):     {((vals > 140) & (vals <= 1400)).sum():>7,}')
    print(f'     >1400   (÷1000):     {(vals > 1400).sum():>7,}')

DIAGNÓSTICO: NOTAS EN PREINSCRIPCIÓN

Registros UJI: 107,218 de 210,986 totales

📊 nota_txt (nota de acceso):
   Rango: [5.0, 11385.0]
   ⚠️ Valores >14: 1,897
   Distribución por escala:
     5-14   (correctos):  105,321
     15-140  (÷10):            94
     141-1400 (÷100):         550
     >1400   (÷1000):       1,253

📊 nota_1 (nota selectividad):
   Rango: [5.0, 9909.0]
   ⚠️ Valores >14: 60,857
   Distribución por escala:
     5-14   (correctos):   46,361
     15-140  (÷10):         3,294
     141-1400 (÷100):      15,481
     >1400   (÷1000):      42,082


In [5]:
# ============================================================================
# CELDA 5: FUNCIÓN DE NORMALIZACIÓN
# ============================================================================
# Las notas en el sistema UJI deben estar en escala [5, 14].
# Algunas vienen multiplicadas por 10, 100 o 1000 (error de carga).
# La función detecta la magnitud y divide para normalizar.
# ============================================================================

def normalizar_nota(s):
    """
    Normaliza notas a escala [5, 14] dividiendo según magnitud.
    
    Reglas:
      - >1400: dividir entre 1000 (ej: 7364 → 7.364)
      - >140 y ≤1400: dividir entre 100 (ej: 736 → 7.36)
      - >14 y ≤140: dividir entre 10 (ej: 73 → 7.3)
      - ≤14: dejar como está
    
    Args:
        s: pd.Series con notas numéricas
    
    Returns:
        pd.Series normalizada (float64)
    """
    s = pd.to_numeric(s, errors='coerce').astype(float).copy()
    s.loc[s > 1400] /= 1000
    s.loc[(s > 140) & (s <= 1400)] /= 100
    s.loc[(s > 14) & (s <= 140)] /= 10
    return s


# --- Verificación ---
test = pd.Series([7.5, 75, 750, 7500, 13.5, 135, 1350, np.nan])
result = normalizar_nota(test)
print('Verificación de normalizar_nota():')
for orig, norm in zip(test, result):
    if pd.notna(orig):
        print(f'   {orig:>8.1f} → {norm:.4f}')
    else:
        print(f'   {"NaN":>8s} → NaN')

assert result.dropna().between(5, 14).all(), 'ERROR: hay valores fuera de [5, 14]'
print('\n✅ Todos los valores normalizados están en [5, 14]')

Verificación de normalizar_nota():
        7.5 → 7.5000
       75.0 → 7.5000
      750.0 → 7.5000
     7500.0 → 7.5000
       13.5 → 13.5000
      135.0 → 13.5000
     1350.0 → 13.5000
        NaN → NaN

✅ Todos los valores normalizados están en [5, 14]


In [6]:
# ============================================================================
# CELDA 6: CORRECCIÓN 1 — NORMALIZAR nota_selectividad EXISTENTE
# ============================================================================
# En df_alumno ya hay 5 valores de nota_selectividad >14 (todos 7364.0).
# Los corregimos antes de la cascada.
# ============================================================================

print('=' * 60)
print('CORRECCIÓN 1: NORMALIZAR VALORES >14 EN df_alumno')
print('=' * 60)

gt14_antes = (df['nota_selectividad'] > 14).sum()
print(f'\nnota_selectividad >14 antes: {gt14_antes}')

if gt14_antes > 0:
    print(f'Valores: {df.loc[df["nota_selectividad"] > 14, "nota_selectividad"].unique()}')
    df['nota_selectividad'] = normalizar_nota(df['nota_selectividad'])
    gt14_despues = (df['nota_selectividad'] > 14).sum()
    print(f'nota_selectividad >14 después: {gt14_despues}')
    print(f'\n✅ Corregidos {gt14_antes} valores')
else:
    print('No hay valores >14 — nada que corregir')

CORRECCIÓN 1: NORMALIZAR VALORES >14 EN df_alumno

nota_selectividad >14 antes: 5
Valores: [7364.]
nota_selectividad >14 después: 0

✅ Corregidos 5 valores


In [7]:
# ============================================================================
# CELDA 7: CORRECCIÓN 2 — CASCADA DESDE PREINSCRIPCIÓN
# ============================================================================
# Para los registros donde nota_acceso o nota_selectividad son NaN,
# rellenamos con las notas de preinscripción:
#   - nota_txt (normalizada) → nota_acceso
#   - nota_1   (normalizada) → nota_selectividad
#
# Merge por: per_id_ficticio + exp_tit_id + curso_aca_ini
# (mismo alumno, misma titulación, mismo año de inicio)
# ============================================================================

print('=' * 60)
print('CORRECCIÓN 2: CASCADA DESDE PREINSCRIPCIÓN')
print('=' * 60)

# --- Preparar preinscripción ---
pre_uji = df_pre[df_pre['uni_id'] == UJI_ID].copy()
pre_uji['nota_txt_norm'] = normalizar_nota(pre_uji['nota_txt'])
pre_uji['nota_1_norm'] = normalizar_nota(pre_uji['nota_1'])

# Renombrar claves de merge
pre_uji = pre_uji.rename(columns={
    'titulacion_centro': 'exp_tit_id',
    'ano': 'curso_aca_ini'
})
pre_uji['curso_aca_ini'] = pre_uji['curso_aca_ini'].astype('Int64')

# Agregar: si hay duplicados por (per_id, tit, curso), tomar el primero
merge_keys = ['per_id_ficticio', 'exp_tit_id', 'curso_aca_ini']
pre_agg = (
    pre_uji
    .groupby(merge_keys)
    .agg(nota_txt_norm=('nota_txt_norm', 'first'),
         nota_1_norm=('nota_1_norm', 'first'))
    .reset_index()
)

print(f'\nRegistros preinscripción UJI: {len(pre_uji):,}')
print(f'Registros únicos (agrupados): {len(pre_agg):,}')

# --- Merge ---
merged = df[merge_keys].merge(pre_agg, on=merge_keys, how='left')
n_match = merged['nota_txt_norm'].notna().sum()
print(f'Matcheados con preinscripción: {n_match:,} de {len(df):,}')

# --- Cascada nota_acceso ---
mask_acceso = df['nota_acceso'].isna() & merged['nota_txt_norm'].notna()
n_rec_acceso = mask_acceso.sum()
df.loc[mask_acceso, 'nota_acceso'] = merged.loc[mask_acceso, 'nota_txt_norm'].values

# --- Cascada nota_selectividad ---
mask_sel = df['nota_selectividad'].isna() & merged['nota_1_norm'].notna()
n_rec_sel = mask_sel.sum()
df.loc[mask_sel, 'nota_selectividad'] = merged.loc[mask_sel, 'nota_1_norm'].values

print(f'\n✅ nota_acceso recuperadas:       +{n_rec_acceso:,}')
print(f'✅ nota_selectividad recuperadas:  +{n_rec_sel:,}')


CORRECCIÓN 2: CASCADA DESDE PREINSCRIPCIÓN



Registros preinscripción UJI: 107,218
Registros únicos (agrupados): 101,730
Matcheados con preinscripción: 82,665 de 109,568

✅ nota_acceso recuperadas:       +1,399
✅ nota_selectividad recuperadas:  +2,564


In [8]:
# ============================================================================
# CELDA 8: CORRECCIÓN 3 — CASCADA INTRA-ALUMNO
# ============================================================================
# Un mismo alumno puede tener nota en unos cursos y no en otros
# (ej: tiene nota_acceso en 2012 pero no en 2013).
# La nota de acceso/selectividad NO cambia entre cursos — es un dato
# fijo del alumno. Podemos propagarla desde cualquier otro registro.
# ============================================================================

print('=' * 60)
print('CORRECCIÓN 3: CASCADA INTRA-ALUMNO')
print('=' * 60)

# --- Identificar alumnos con nota en algún registro ---
sin_acceso = df[df['nota_acceso'].isna()]
sin_sel = df[df['nota_selectividad'].isna()]

# Para nota_acceso: buscar alumnos que SÍ tienen nota en otros registros
alumnos_sin_acceso = sin_acceso['per_id_ficticio'].unique()
ref_acceso = (
    df[df['per_id_ficticio'].isin(alumnos_sin_acceso) & df['nota_acceso'].notna()]
    .groupby('per_id_ficticio')['nota_acceso']
    .first()
    .rename('nota_acceso_ref')
)

# Para nota_selectividad
alumnos_sin_sel = sin_sel['per_id_ficticio'].unique()
ref_sel = (
    df[df['per_id_ficticio'].isin(alumnos_sin_sel) & df['nota_selectividad'].notna()]
    .groupby('per_id_ficticio')['nota_selectividad']
    .first()
    .rename('nota_sel_ref')
)

print(f'\nAlumnos sin nota_acceso: {len(alumnos_sin_acceso):,}')
print(f'  De ellos, {len(ref_acceso):,} tienen nota en otros registros')
print(f'\nAlumnos sin nota_selectividad: {len(alumnos_sin_sel):,}')
print(f'  De ellos, {len(ref_sel):,} tienen nota en otros registros')

# --- Aplicar propagación ---
df = df.merge(ref_acceso, on='per_id_ficticio', how='left')
df = df.merge(ref_sel, on='per_id_ficticio', how='left')

mask_a = df['nota_acceso'].isna() & df['nota_acceso_ref'].notna()
mask_s = df['nota_selectividad'].isna() & df['nota_sel_ref'].notna()

n_intra_acceso = mask_a.sum()
n_intra_sel = mask_s.sum()

df.loc[mask_a, 'nota_acceso'] = df.loc[mask_a, 'nota_acceso_ref']
df.loc[mask_s, 'nota_selectividad'] = df.loc[mask_s, 'nota_sel_ref']

# Limpiar columnas temporales
df = df.drop(columns=['nota_acceso_ref', 'nota_sel_ref'])

print(f'\n✅ nota_acceso propagadas intra-alumno:       +{n_intra_acceso:,}')
print(f'✅ nota_selectividad propagadas intra-alumno:  +{n_intra_sel:,}')

CORRECCIÓN 3: CASCADA INTRA-ALUMNO

Alumnos sin nota_acceso: 3,158
  De ellos, 665 tienen nota en otros registros

Alumnos sin nota_selectividad: 3,758
  De ellos, 728 tienen nota en otros registros



✅ nota_acceso propagadas intra-alumno:       +1,754
✅ nota_selectividad propagadas intra-alumno:  +1,820


In [9]:
# ============================================================================
# CELDA 9: VALIDACIÓN
# ============================================================================
# Verificar que todas las notas están en rango válido [5, 14]
# y que no se han creado registros nuevos ni perdido columnas.
# ============================================================================

print('=' * 60)
print('VALIDACIÓN')
print('=' * 60)

# --- Rangos ---
for col in ['nota_acceso', 'nota_selectividad']:
    vals = df[col].dropna()
    vmin, vmax = vals.min(), vals.max()
    gt14 = (vals > 14).sum()
    lt5 = (vals < 5).sum()
    print(f'\n{col}:')
    print(f'   Rango: [{vmin:.4f}, {vmax:.4f}]')
    print(f'   >14: {gt14} | <5: {lt5}')
    assert gt14 == 0, f'ERROR: {col} tiene {gt14} valores >14'
    assert lt5 == 0, f'ERROR: {col} tiene {lt5} valores <5'

# --- Integridad ---
df_orig = pd.read_parquet(path_alumno)
assert len(df) == len(df_orig), f'ERROR: filas cambiaron ({len(df_orig)} → {len(df)})'
assert list(df.columns) == list(df_orig.columns), 'ERROR: columnas cambiaron'

print(f'\n✅ Registros: {len(df):,} (sin cambios)')
print(f'✅ Columnas: {len(df.columns)} (sin cambios)')
print(f'✅ Todos los valores en rango [5, 14]')
print(f'✅ Validación superada')

VALIDACIÓN

nota_acceso:
   Rango: [5.0000, 13.8050]
   >14: 0 | <5: 0

nota_selectividad:
   Rango: [5.0000, 12.5360]
   >14: 0 | <5: 0



✅ Registros: 109,568 (sin cambios)
✅ Columnas: 37 (sin cambios)
✅ Todos los valores en rango [5, 14]
✅ Validación superada


In [10]:
# ============================================================================
# CELDA 10: COMPARACIÓN ANTES vs DESPUÉS
# ============================================================================

print('=' * 60)
print('COMPARACIÓN ANTES vs DESPUÉS')
print('=' * 60)

n = len(df)
despues = {
    'nota_acceso_nan': df['nota_acceso'].isna().sum(),
    'nota_selectividad_nan': df['nota_selectividad'].isna().sum(),
    'nota_selectividad_gt14': (df['nota_selectividad'] > 14).sum(),
    'sin_ninguna': (df['nota_acceso'].isna() & df['nota_selectividad'].isna()).sum()
}

print(f'\n{"MÉTRICA":<35s} {"ANTES":>10s} {"DESPUÉS":>10s} {"GANANCIA":>10s}')
print('-' * 67)

for key, label in [
    ('nota_acceso_nan', 'nota_acceso NaN'),
    ('nota_selectividad_nan', 'nota_selectividad NaN'),
    ('nota_selectividad_gt14', 'nota_selectividad >14'),
    ('sin_ninguna', 'Sin NINGUNA nota'),
]:
    a = antes[key]
    d = despues[key]
    ganancia = a - d
    print(f'{label:<35s} {a:>10,} {d:>10,} {ganancia:>+10,}')

# Desglose de la ganancia
print(f'\n📊 Desglose de recuperaciones:')
print(f'   Cascada preinscripción:  nota_acceso +{n_rec_acceso:,}, nota_selectividad +{n_rec_sel:,}')
print(f'   Cascada intra-alumno:    nota_acceso +{n_intra_acceso:,}, nota_selectividad +{n_intra_sel:,}')

# Rangos
print(f'\n📊 Rangos:')
for col in ['nota_acceso', 'nota_selectividad']:
    v = df[col].dropna()
    print(f'   {col}: [{v.min():.4f}, {v.max():.4f}]')

COMPARACIÓN ANTES vs DESPUÉS

MÉTRICA                                  ANTES    DESPUÉS   GANANCIA
-------------------------------------------------------------------
nota_acceso NaN                         10,973      7,820     +3,153
nota_selectividad NaN                   14,254      9,870     +4,384
nota_selectividad >14                        5          0         +5
Sin NINGUNA nota                        10,747      7,771     +2,976

📊 Desglose de recuperaciones:
   Cascada preinscripción:  nota_acceso +1,399, nota_selectividad +2,564
   Cascada intra-alumno:    nota_acceso +1,754, nota_selectividad +1,820

📊 Rangos:
   nota_acceso: [5.0000, 13.8050]
   nota_selectividad: [5.0000, 12.5360]


In [11]:
# ============================================================================
# CELDA 11: DOCUMENTAR LOS QUE QUEDAN SIN NOTA
# ============================================================================

print('=' * 60)
print('REGISTROS SIN NOTA — ANÁLISIS')
print('=' * 60)

sin = df[df['nota_acceso'].isna() & df['nota_selectividad'].isna()]
n_sin = len(sin)
n_alumnos_sin = sin['per_id_ficticio'].nunique()

print(f'\nRegistros sin ninguna nota: {n_sin:,} ({n_sin/n*100:.1f}%)')
print(f'Alumnos únicos sin nota: {n_alumnos_sin:,}')

# --- Por tipo de alumno ---
print(f'\nPor tipo (nuevo S/N):')
print(sin['nuevo'].value_counts().to_string())
pct_no_nuevo = (sin['nuevo'] == 'N').sum() / n_sin * 100
print(f'→ {pct_no_nuevo:.0f}% son alumnos NO nuevos (adaptaciones, traslados, etc.)')

# --- Por curso ---
print(f'\nPor curso_aca_ini:')
print(sin['curso_aca_ini'].value_counts().sort_index().to_string())
pct_2009_2013 = sin[sin['curso_aca_ini'].isin([2009, 2010, 2011, 2012, 2013])].shape[0] / n_sin * 100
print(f'→ {pct_2009_2013:.0f}% concentrados en 2009-2013 (implantación de Bolonia)')

# --- Por vía de acceso ---
tiene_via = sin['via_acceso'].notna().sum()
print(f'\nCon vía de acceso: {tiene_via} de {n_sin}')
if tiene_via > 0:
    print(sin['via_acceso'].value_counts().to_string())

print(f'\n📝 Conclusión: Los {n_sin:,} registros sin nota son mayoritariamente')
print(f'   alumnos NO nuevos de 2009-2013 (adaptación a grado Bolonia).')
print(f'   No tienen nota de acceso porque entraron por vías que no la requieren.')
print(f'   Quedan como NaN documentado — no es un error, es dato ausente legítimo.')

REGISTROS SIN NOTA — ANÁLISIS

Registros sin ninguna nota: 7,771 (7.1%)
Alumnos únicos sin nota: 2,481

Por tipo (nuevo S/N):
nuevo
N    7680
S      91
→ 99% son alumnos NO nuevos (adaptaciones, traslados, etc.)

Por curso_aca_ini:
curso_aca_ini
2009    2197
2010     870
2011     561
2012    1577
2013    1124
2014     510
2015     349
2016     178
2017     172
2018     141
2019      66
2020      26
→ 81% concentrados en 2009-2013 (implantación de Bolonia)

Con vía de acceso: 190 de 7771
via_acceso
Pruebas acceso Bachiller Logse                 171
Ciclo Formativo de Grado sup. o equivalente     10
Extranjeros CEE                                  6
Titulados Universitarios                         3

📝 Conclusión: Los 7,771 registros sin nota son mayoritariamente
   alumnos NO nuevos de 2009-2013 (adaptación a grado Bolonia).
   No tienen nota de acceso porque entraron por vías que no la requieren.
   Quedan como NaN documentado — no es un error, es dato ausente legítimo.


In [12]:
# ============================================================================
# CELDA 12: GUARDAR DATASET
# ============================================================================
# Sobreescribe df_alumno.parquet y df_alumno.csv con las notas corregidas.
# El pipeline posterior (Fase 2, 3, 4...) lee estos archivos sin cambios.
# ============================================================================

print('=' * 60)
print('GUARDANDO DATASET')
print('=' * 60)

# --- Parquet ---
path_parquet = RUTA_PROCESSED / 'df_alumno.parquet'
df.to_parquet(path_parquet, index=False)
size_parquet = path_parquet.stat().st_size / 1024 / 1024
print(f'\n✅ {path_parquet.name}: {size_parquet:.2f} MB')

# --- CSV ---
path_csv = RUTA_PROCESSED / 'df_alumno.csv'
df.to_csv(path_csv, index=False, sep=';', decimal=',')
size_csv = path_csv.stat().st_size / 1024 / 1024
print(f'✅ {path_csv.name}: {size_csv:.2f} MB')

GUARDANDO DATASET



✅ df_alumno.parquet: 2.27 MB


✅ df_alumno.csv: 30.75 MB


In [13]:
# ============================================================================
# CELDA 13: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F1-M04c: CORRECCIÓN DE NOTAS — COMPLETADO')
print('=' * 60)

print(f'''
Correcciones aplicadas:

  1. Normalización:  {antes['nota_selectividad_gt14']} valores >14 en nota_selectividad → corregidos
  2. Cascada preinscripción:  nota_acceso +{n_rec_acceso:,}  |  nota_selectividad +{n_rec_sel:,}
  3. Cascada intra-alumno:    nota_acceso +{n_intra_acceso:,}  |  nota_selectividad +{n_intra_sel:,}

Resultado:
  nota_acceso NaN:       {antes['nota_acceso_nan']:,} → {despues['nota_acceso_nan']:,} (recuperados +{antes['nota_acceso_nan'] - despues['nota_acceso_nan']:,})
  nota_selectividad NaN: {antes['nota_selectividad_nan']:,} → {despues['nota_selectividad_nan']:,} (recuperados +{antes['nota_selectividad_nan'] - despues['nota_selectividad_nan']:,})
  Sin ninguna nota:      {antes['sin_ninguna']:,} → {despues['sin_ninguna']:,} (NaN documentado)

Archivos actualizados:
  data/02_processed/df_alumno.parquet
  data/02_processed/df_alumno.csv

📌 Siguiente: f1_m04d (corrección vía de acceso — P5)
''')


✅ F1-M04c: CORRECCIÓN DE NOTAS — COMPLETADO

Correcciones aplicadas:

  1. Normalización:  5 valores >14 en nota_selectividad → corregidos
  2. Cascada preinscripción:  nota_acceso +1,399  |  nota_selectividad +2,564
  3. Cascada intra-alumno:    nota_acceso +1,754  |  nota_selectividad +1,820

Resultado:
  nota_acceso NaN:       10,973 → 7,820 (recuperados +3,153)
  nota_selectividad NaN: 14,254 → 9,870 (recuperados +4,384)
  Sin ninguna nota:      10,747 → 7,771 (NaN documentado)

Archivos actualizados:
  data/02_processed/df_alumno.parquet
  data/02_processed/df_alumno.csv

📌 Siguiente: f1_m04d (corrección vía de acceso — P5)

